## My Personal RAG App

A chatbot that answers questions using information from my personal documents and online profiles.

Load the environment variables from the `.env` file so the application can access the API key securely.


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

Connect to the OpenRouter API and initialize the language model that will generate answers.

In [4]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0
)

## RAG

These are the main steps for building the vector database: loading documents, splitting them into chunks, creating embeddings, and storing them in FAISS.

In [5]:
## step 1 : create vector database

# Loading
# Chunking
# Embedding
# Vector stores


# Loading
import glob
from langchain_classic.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader

documents = []

# pdf files
for path in glob.glob("data/*.pdf"):
    docs = PyPDFLoader(path).load()
    for d in docs:
        d.metadata["source"] = os.path.basename(path)
    documents.extend(docs)

# txt files
for path in glob.glob("data/*.txt") + glob.glob("data/*.md"):
    docs = TextLoader(path, encoding="utf-8").load()
    for d in docs:
        d.metadata["source"] = os.path.basename(path)
    documents.extend(docs)

# links (github, Linkedin) 
links = [
    "https://github.com/dalabilal/Toxic-content-classification-project/blob/master/README.md",
    "https://github.com/dalabilal/FE-Student-Accommodation/blob/main/README.md",
    "https://www.linkedin.com/in/dalah-hashlamoon-59b7bb1b6/"   
]

for url in links:
    docs = WebBaseLoader(url).load()
    for d in docs:
        d.metadata["source"] = url
    documents.extend(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
for d in documents:
    if d.metadata["source"].startswith("http"):
        text = " ".join(d.page_content.split())
        print(d.metadata["source"])
        print(len(text), "characters")
        print(text[:200])
        print("-----")

https://github.com/dalabilal/Toxic-content-classification-project/blob/master/README.md
4015 characters
Toxic-content-classification-project/README.md at master · dalabilal/Toxic-content-classification-project · GitHub Skip to content Navigation MenuSign inAppearance settingsPlatformAI CODE CREATIONGitH
-----
https://github.com/dalabilal/FE-Student-Accommodation/blob/main/README.md
5581 characters
FE-Student-Accommodation/README.md at main · dalabilal/FE-Student-Accommodation · GitHub Skip to content Navigation MenuSign inAppearance settingsPlatformAI CODE CREATIONGitHub CopilotWrite better cod
-----
https://www.linkedin.com/in/dalah-hashlamoon-59b7bb1b6/
0 characters

-----


Display the number of loaded documents and their source names to verify that the data was collected successfully


In [7]:
print(len(documents))

for i in documents:
    print(i.metadata["source"])

8
Dalah_Hashlamoon_CE.pdf
ML & Data Science.pdf
intoduction.txt
Capstone-project.md
proposal-of-capstone.md
https://github.com/dalabilal/Toxic-content-classification-project/blob/master/README.md
https://github.com/dalabilal/FE-Student-Accommodation/blob/main/README.md
https://www.linkedin.com/in/dalah-hashlamoon-59b7bb1b6/


Split the documents into smaller text chunks so the system can retrieve the most relevant information for each question


In [8]:
# Chunking
from langchain_classic.text_splitter import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
)

chunks = splitter.split_documents(documents)

print(len(chunks))

58


Convert the text chunks into numerical embeddings and store them in FAISS for similarity-based retrieval

In [9]:
# Embedding
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)


# vector store
from langchain_classic.vectorstores import FAISS

vectorDB = FAISS.from_documents(chunks, embeddings)

C:\Users\smart\AppData\Local\Temp\ipykernel_22376\2166873027.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\smart\Desktop\Cellulla-3week-task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4761.84it/s]


Save the FAISS index locally so it can be reused without rebuilding it every time

In [10]:
# save the database so i don't build it again
vectorDB.save_local("faiss_index")

In [ ]:
# next time just i load it no need to run the loading and chunking again

# vectorDB = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

## Step 2: Search

Search the vector database to find document chunks that are semantically similar to a question

Search for the three most relevant chunks for a sample question and display their similarity scores, sources, and text


In [9]:
query = "what are my skills"

similar_docs = vectorDB.similarity_search_with_score(query, k=3)

for i in similar_docs:
    print(i[1])
    print(i[0].metadata["source"])
    print(i[0].page_content)
    print("-----")

1.2956424
Dalah_Hashlamoon_CE.pdf
, and predictive analysis techniques.
– Gained practical experience with Python libraries and machine learning fundamentals.
TECHNICAL SKILLS
Programming Languages : Python, JavaScript, TypeScript, C++
Web Development : React.js, Express.js, Node.js, Tailwind CSS, Bootstrap
Databases : SQL, NoSQL (MongoDB, Redis, neo4j)
AI & Machine Learning : TensorFlow, Scikit-learn, Pandas, NumPy
DevOps & Tools : Git, Azure, Postman, Jira
Methodologies : Agile (Scrum)
EDUCATION
Master’s Degree in Intelligent Systems and Data Science - Palestine Polytechnic University Sep 2025 – Present
BS in Computer Engineering - Information Security at Palestine Polytechnic University Sep 2020 – Jun 2025
PROJECTS
Student Accommodation (Security graduation project) | MERN Full Stack Dec 2023 – Mar 2024

-----
1.4484452
Dalah_Hashlamoon_CE.pdf
end Development: Built responsive and dynamic user interfaces using React.js, ensuring cross-browser compatibility
and optimized performance.

the score is the distance, so **smaller = closer**. if the score is big the chunk is not related to the question

## step 3 : prompt template

Create a reusable prompt that includes the retrieved context and the user's question, instructing the model not to guess or use information outside the context.

In [10]:
from langchain_core.prompts import PromptTemplate

temp = """
you are assistant that answer questions about one person, Answer the next Question using provided context only,
If the answer is not in the context, just say you don't know.
don't guess and don't use any information outside the context.
answer should be within 200 words or lower only
at the end write the sources you used

## context :
{context}

## Question :
{Question}
"""

temp = PromptTemplate.from_template(temp)

Retrieve the four closest chunks, keep only those below the chosen distance threshold, collect their sources, and insert the results into the prompt.

In [11]:
query = "Dalah participated in the AI Hackathon U-TEACH LEAGUE?"

# test questions
# what are my skills
# what is my education
# what certificates do i have
# where did i work
# what is my favorite football team   <-- not in my files, it should say i don't know

similar_docs = vectorDB.similarity_search_with_score(query, k=4)

context = []
sources = []

for i in similar_docs:
    # keep only the chunks that are close to the question
    if i[1] < 1.2:
        context.append("[source: " + i[0].metadata["source"] + "]\n" + i[0].page_content)
        sources.append(i[0].metadata["source"])

prompt = temp.format(
    context="\n\n".join(context),
    Question=query
)

print(prompt)


you are assistant that answer questions about one person, Answer the next Question using provided context only,
If the answer is not in the context, just say you don't know.
don't guess and don't use any information outside the context.
answer should be within 200 words or lower only
at the end write the sources you used

## context :
[source: intoduction.txt]
Dala Hashlamoon is a Computer Engineering graduate specializing in Information Security, with a strong interest in software development, artificial intelligence, and problem-solving. In June 2024, she participated in the AI Hackathon U-TEACH LEAGUE with Microsoft and Finomena, where she collaborated with a university team to develop a chatbot website for Jawwal using LangChain and Azure OpenAI. After the elimination stage, the team was restructured into six members and achieved second place as a support team. In 2025, she also participated in a Quantum Computing Program in Palestine, where she explored quantum computing applicat

If no relevant context is found, return a fallback message; otherwise, send the prompt to the language model and retrieve its answer

In [12]:
if len(context) == 0:
    response = "I couldn't find that in my sources"
else:
    response = llm.invoke(prompt).content

response

'Yes, Dala Hashlamoon participated in the AI Hackathon U-TEACH LEAGUE in June 2024 with Microsoft and Finomena.\n\nSources:\n[source: intoduction.txt]'

In [13]:
from IPython.display import display , Markdown

display(Markdown(response))

Yes, Dala Hashlamoon participated in the AI Hackathon U-TEACH LEAGUE in June 2024 with Microsoft and Finomena.

Sources:
[source: intoduction.txt]

In [14]:
print(set(sources))

{'intoduction.txt'}


to ask another question: change `query` in the cell above and run the cells again